# Notebook 05 — Combinar DataFrames (`merge` y `concat`)

En el mundo real los datos casi nunca viven en una sola tabla. Una empresa tiene una tabla de **clientes**, otra de **pedidos**, otra de **productos**... y para responder preguntas interesantes hay que **unirlas**. 🔗

Hoy aprendes las dos formas principales de combinar `DataFrame`s en pandas:

- **`pd.concat`** — apilar tablas (juntar filas que comparten columnas).
- **`pd.merge`** (o `df.merge`) — unir tablas por una **clave común** (como un `JOIN` de SQL o un `VLOOKUP` de Excel).

## Objetivos de aprendizaje

1. Apilar `DataFrame`s con `pd.concat()`.
2. Unir dos `DataFrame`s por una clave con `merge` (`how="inner"`).
3. Entender los tipos de join: `inner`, `left`, `right`, `outer`.
4. Usar `left_on` y `right_on` cuando las claves se llaman distinto en cada tabla.
5. Diagnosticar resultados de un merge mirando la **shape** y los NaN.

---

## 1. Repaso rápido del Notebook 04

| Operación | Sintaxis |
|---|---|
| Una agregación | `df.groupby("col")["v"].mean()` |
| Varias agregaciones | `df.groupby("col")["v"].agg(["min","max","mean"])` |
| Multi-columna | `df.groupby(["c1","c2"])["v"].mean()` |
| Conteo categórico | `df["col"].value_counts()` |

---

## 2. Setup

Creamos dos `DataFrame`s pequeños **a mano** para entender la mecánica antes de pasar a un dataset real.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

employees = pd.DataFrame({
    "employee_id": [1, 2, 3, 4, 5, 6],
    "name": ["Alice", "Bob", "Carol", "David", "Eve", "Frank"],
    "dept_id": [10, 20, 10, 30, 20, 99],   # employee 6 has an unknown department
})

departments = pd.DataFrame({
    "dept_id": [10, 20, 30, 40],            # department 40 has no employees yet
    "dept_name": ["Engineering", "Sales", "HR", "Finance"],
    "city": ["Madrid", "Barcelona", "Sevilla", "Bilbao"],
})

print("=== employees ===")
print(employees)
print("\n=== departments ===")
print(departments)

Observa dos cosas que harán que veamos las distintas variantes de `merge` con claridad:

- **Frank** (employee_id 6) está en el departamento `99`, que **no existe** en `departments`.
- **Finance** (dept_id 40) **no tiene ningún empleado** asignado.

Estas asimetrías son exactamente lo que un `merge` debe manejar bien.

---

## 3. `pd.concat` — apilar filas

Cuando tienes dos `DataFrame`s con **las mismas columnas** y quieres juntarlos en uno solo (ej. los pedidos de enero y los de febrero), usas `pd.concat`:

```python
pd.concat([df1, df2], ignore_index=True)
```

`ignore_index=True` reinicia el índice para que no aparezcan duplicados (0, 1, 0, 1, ...).

### Demo

In [ ]:
# Split employees into two halves and concat them back
half_a = employees.iloc[:3]
half_b = employees.iloc[3:]
print("half_a shape:", half_a.shape)
print("half_b shape:", half_b.shape)

reunited = pd.concat([half_a, half_b], ignore_index=True)
print("\nreunited:")
print(reunited)

### 🏋️ Ejercicio 1 — `pd.concat`

Crea dos `DataFrame`s de empleados nuevos y concaténalos.

```python
new_hires_q1 = pd.DataFrame({
    "employee_id": [7, 8],
    "name": ["Gina", "Hugo"],
    "dept_id": [20, 10],
})

new_hires_q2 = pd.DataFrame({
    "employee_id": [9, 10, 11],
    "name": ["Iris", "Jon", "Kira"],
    "dept_id": [10, 30, 20],
})
```

Crea un `DataFrame` llamado **`all_new_hires`** que sea la concatenación de `new_hires_q1` y `new_hires_q2` con el índice reseteado.

In [ ]:
new_hires_q1 = pd.DataFrame({
    "employee_id": [7, 8],
    "name": ["Gina", "Hugo"],
    "dept_id": [20, 10],
})

new_hires_q2 = pd.DataFrame({
    "employee_id": [9, 10, 11],
    "name": ["Iris", "Jon", "Kira"],
    "dept_id": [10, 30, 20],
})

# YOUR CODE HERE
all_new_hires = None


In [ ]:
# Tests
assert isinstance(all_new_hires, pd.DataFrame), "all_new_hires must be a DataFrame"
assert all_new_hires.shape == (5, 3), f"Expected shape (5, 3), got {all_new_hires.shape}"
assert list(all_new_hires.columns) == ["employee_id", "name", "dept_id"], \
    f"Columns must be ['employee_id', 'name', 'dept_id'], got {list(all_new_hires.columns)}"
assert list(all_new_hires.index) == [0, 1, 2, 3, 4], \
    f"Index must be reset to 0..4, got {list(all_new_hires.index)}"
assert list(all_new_hires["name"]) == ["Gina", "Hugo", "Iris", "Jon", "Kira"], \
    "Names mismatch — order should be q1 then q2"

print("✅ ¡Bien! Apilaste los dos DataFrames correctamente.")
all_new_hires

---

## 4. `merge` — unir dos tablas por una clave

Aquí está la operación estrella. Tienes dos tablas que comparten una columna en común (la **clave**), y quieres juntarlas en una sola.

```python
df1.merge(df2, on="clave", how="inner")
```

- `on="clave"` → la columna que sirve para emparejar filas.
- `how="..."` → el **tipo de join** (ver la siguiente sección).

### Tipos de join

```
  inner    →   solo filas que existan en AMBAS tablas
  left     →   TODAS las filas de la izquierda  (las no emparejadas tendrán NaN en columnas de la derecha)
  right    →   TODAS las filas de la derecha    (las no emparejadas tendrán NaN en columnas de la izquierda)
  outer    →   TODAS las filas de ambas         (NaN donde falten datos)
```

> 💡 Analogías:
> - `merge` ≈ `JOIN` de SQL.
> - `merge` ≈ `VLOOKUP` de Excel pero **bidireccional** y mucho más potente.

### Demo — `inner`

In [ ]:
# Inner merge: only rows that match on dept_id in BOTH tables
employees.merge(departments, on="dept_id", how="inner")

👀 Observa que **Frank desapareció** (su `dept_id=99` no existe en `departments`) y **Finance también** (`dept_id=40` no tiene empleados). Esto es exactamente lo que hace un `inner`.

### 🏋️ Ejercicio 2 — `merge` inner

Crea un `DataFrame` llamado **`employees_with_dept`** que sea el resultado de hacer un **inner merge** entre `employees` y `departments` usando `dept_id` como clave.

In [ ]:
# YOUR CODE HERE
employees_with_dept = None


In [ ]:
# Tests
assert isinstance(employees_with_dept, pd.DataFrame), "employees_with_dept must be a DataFrame"
assert employees_with_dept.shape == (5, 5), f"Expected shape (5, 5), got {employees_with_dept.shape}"
assert "dept_name" in employees_with_dept.columns, "Result must include 'dept_name'"
assert "city" in employees_with_dept.columns, "Result must include 'city'"
assert "Frank" not in set(employees_with_dept["name"]), \
    "Frank should NOT be in the inner merge (his dept_id=99 doesn't exist)"
assert "Finance" not in set(employees_with_dept["dept_name"]), \
    "Finance should NOT be in the inner merge (no employee has dept_id=40)"
alice_dept = employees_with_dept.loc[employees_with_dept["name"] == "Alice", "dept_name"].iloc[0]
assert alice_dept == "Engineering", f"Alice should be in Engineering, got {alice_dept}"

print("✅ ¡Excelente! Hiciste tu primer inner merge.")
employees_with_dept

---

## 5. `left` vs `inner` — conservar todas las filas de la izquierda

A veces **no quieres perder filas**. Por ejemplo: la tabla `employees` es tu fuente de verdad y aunque haya un `dept_id` desconocido, quieres conservar a Frank.

```python
employees.merge(departments, on="dept_id", how="left")
```

Las columnas que vienen de `departments` aparecerán como **NaN** para los empleados sin match.

### Demo

In [ ]:
# Left merge: keep ALL employees, fill missing department info with NaN
employees.merge(departments, on="dept_id", how="left")

👀 Frank aparece, pero con `NaN` en `dept_name` y `city`. Eso es la firma de un `left` join cuando hay claves huérfanas.

### 🏋️ Ejercicio 3 — `left` merge

Crea un `DataFrame` llamado **`employees_left`** haciendo un **left merge** de `employees` con `departments` por `dept_id`. Después, calcula cuántos `NaN` aparecen en la columna `dept_name` y guárdalo en un entero llamado **`unmatched_count`**.

In [ ]:
# YOUR CODE HERE — assign both variables
employees_left = None
unmatched_count = None


In [ ]:
# Tests
assert isinstance(employees_left, pd.DataFrame), "employees_left must be a DataFrame"
assert employees_left.shape == (6, 5), f"Expected shape (6, 5), got {employees_left.shape}"
assert "Frank" in set(employees_left["name"]), "Frank must be present in a left merge"
frank_row = employees_left.loc[employees_left["name"] == "Frank"].iloc[0]
assert pd.isna(frank_row["dept_name"]), "Frank's dept_name must be NaN"
assert pd.isna(frank_row["city"]), "Frank's city must be NaN"

assert isinstance(unmatched_count, (int, np.integer)), \
    f"unmatched_count must be an integer, got {type(unmatched_count).__name__}"
assert unmatched_count == 1, f"Expected 1 unmatched row, got {unmatched_count}"

print("✅ ¡Bien! Conservaste todas las filas y detectaste 1 empleado sin departamento.")
employees_left

---

## 6. Claves con **nombres distintos** en cada tabla

A veces la columna de la clave **se llama distinto** en cada tabla (muy común en bases de datos reales). Para eso existen `left_on` y `right_on`:

```python
df1.merge(df2, left_on="clave_izq", right_on="clave_der", how="inner")
```

### Demo

In [ ]:
# Rename dept_id in employees to 'department' to simulate the situation
employees_renamed = employees.rename(columns={"dept_id": "department"})
print(employees_renamed.head(2))

# Now the keys have different names — use left_on / right_on
employees_renamed.merge(
    departments,
    left_on="department",
    right_on="dept_id",
    how="inner",
)

### 🏋️ Ejercicio 4 — `left_on` / `right_on`

Las tablas siguientes tienen las **mismas claves** pero con **nombres distintos**:

```python
products = pd.DataFrame({
    "product_id": [101, 102, 103, 104],
    "product_name": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "category_code": ["E1", "E2", "E2", "E1"],
})

categories = pd.DataFrame({
    "code": ["E1", "E2", "E3"],
    "category_label": ["Computers", "Accessories", "Cables"],
})
```

Crea un `DataFrame` llamado **`products_with_category`** que una `products` y `categories` por su clave (un `inner` merge). La clave en `products` se llama `category_code`; en `categories` se llama `code`.

In [ ]:
products = pd.DataFrame({
    "product_id": [101, 102, 103, 104],
    "product_name": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "category_code": ["E1", "E2", "E2", "E1"],
})

categories = pd.DataFrame({
    "code": ["E1", "E2", "E3"],
    "category_label": ["Computers", "Accessories", "Cables"],
})

# YOUR CODE HERE
products_with_category = None


In [ ]:
# Tests
assert isinstance(products_with_category, pd.DataFrame), "products_with_category must be a DataFrame"
assert products_with_category.shape[0] == 4, \
    f"Expected 4 rows (all products match a category), got {products_with_category.shape[0]}"
assert "category_label" in products_with_category.columns, \
    "Result must include 'category_label' from the categories table"
assert "product_name" in products_with_category.columns, \
    "Result must include 'product_name' from the products table"

laptop_label = products_with_category.loc[
    products_with_category["product_name"] == "Laptop", "category_label"
].iloc[0]
assert laptop_label == "Computers", f"Laptop should map to 'Computers', got '{laptop_label}'"

mouse_label = products_with_category.loc[
    products_with_category["product_name"] == "Mouse", "category_label"
].iloc[0]
assert mouse_label == "Accessories", f"Mouse should map to 'Accessories', got '{mouse_label}'"

assert "Cables" not in set(products_with_category["category_label"]), \
    "'Cables' has no products — it shouldn't appear in an inner merge"

print("✅ ¡Genial! Uniste tablas con claves de nombres distintos.")
products_with_category

---

## 7. Caso realista — partir y reunir el dataset `tips`

Para terminar, vamos a simular un caso típico: tienes la información de las cuentas en una tabla y los **datos de contexto** del cliente en otra. Tu trabajo es **unirlas**.

Cargamos `tips` (de seaborn) y lo dividimos en dos sub-tablas con un identificador común `bill_id`.

In [ ]:
tips = sns.load_dataset("tips")
tips = tips.reset_index(drop=True)
tips["bill_id"] = tips.index   # synthetic primary key

# Split into two related tables (as if they came from different sources)
tips_billing = tips[["bill_id", "total_bill", "tip", "smoker", "day"]].copy()
tips_context = tips[["bill_id", "sex", "time", "size"]].copy()

print("tips_billing:")
print(tips_billing.head(3))
print(f"\nshape: {tips_billing.shape}")
print("\ntips_context:")
print(tips_context.head(3))
print(f"\nshape: {tips_context.shape}")

### 🏋️ Ejercicio 5 — reunir las dos tablas

Crea un `DataFrame` llamado **`tips_full`** que sea el merge de `tips_billing` y `tips_context` por la clave **`bill_id`** (usa un `inner` merge). El resultado debe tener todas las columnas originales del dataset `tips`.

In [ ]:
# YOUR CODE HERE
tips_full = None


In [ ]:
# Tests
expected_cols = {"total_bill", "tip", "sex", "smoker", "day", "time", "size"}

assert isinstance(tips_full, pd.DataFrame), "tips_full must be a DataFrame"
assert tips_full.shape[0] == 244, f"Expected 244 rows (same as tips), got {tips_full.shape[0]}"
assert expected_cols.issubset(tips_full.columns), \
    f"Missing columns: {expected_cols - set(tips_full.columns)}"
assert "bill_id" in tips_full.columns, "bill_id must be preserved as the join key"

# Spot-check: row 0 in the recovered table should match row 0 in the original tips
original_row = tips.iloc[0]
recovered_row = tips_full.loc[tips_full["bill_id"] == 0].iloc[0]
assert np.isclose(recovered_row["total_bill"], original_row["total_bill"]), \
    "Row 0 total_bill mismatch — keys did not align"
assert recovered_row["sex"] == original_row["sex"], "Row 0 sex mismatch"
assert recovered_row["time"] == original_row["time"], "Row 0 time mismatch"

print(f"✅ ¡Excelente! Reuniste las dos tablas en un DataFrame de {tips_full.shape[0]} filas.")
tips_full.head()

---

## 8. Resumen — ¿qué aprendiste?

🎉 ¡Bien hecho! Ahora puedes combinar datos de múltiples fuentes.

| Operación | Sintaxis |
|---|---|
| Apilar filas | `pd.concat([df1, df2], ignore_index=True)` |
| Unir por clave (intersección) | `df1.merge(df2, on="k", how="inner")` |
| Unir conservando izquierda | `df1.merge(df2, on="k", how="left")` |
| Unir conservando derecha | `df1.merge(df2, on="k", how="right")` |
| Unir conservando ambas | `df1.merge(df2, on="k", how="outer")` |
| Claves con distinto nombre | `df1.merge(df2, left_on="k1", right_on="k2")` |

### Reglas prácticas

1. **Mira la `shape`** antes y después del merge — si no cuadra, hay duplicados o claves huérfanas.
2. **Cuenta los NaN** después de un `left` o `outer` para detectar matches faltantes.
3. **Empieza con `inner`** y pasa a `left`/`outer` solo si necesitas conservar filas.
4. Si una clave tiene **valores duplicados** en ambas tablas, el merge crea **una fila por cada combinación** — cuidado con explosiones de tamaño.

## ¿Qué viene en el próximo notebook?

En **Notebook 06 — Mini proyecto EDA** vas a poner en práctica **todo** lo aprendido (NB01 → NB05) sobre un dataset nuevo y famoso: el **Titanic** 🚢. Sin nuevos conceptos, solo aplicación. ¡Nos vemos ahí!